# Telco Customer Churn - Exploratory Data Analysis

**Author:** Data Scientist  
**Date:** 2026-01-07  
**Dataset:** [Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

---

## Objective

Perform comprehensive exploratory data analysis on the Telco Customer Churn dataset to:
- Understand data quality and structure
- Analyze churn patterns and rates
- Identify key features influencing customer churn
- Generate actionable insights for feature engineering and modeling

---

## 1. Setup & Imports

In [ ]:
# Standard library imports
import sys
from pathlib import Path
import warnings
import os

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical analysis
from scipy import stats
from scipy.stats import chi2_contingency

# Add src to path for custom modules
# Detect project root (parent of notebooks directory)
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    project_root = notebook_dir.parent
else:
    # If running from project root, use current directory
    project_root = notebook_dir

src_path = project_root / 'src'
sys.path.insert(0, str(src_path))

# Custom modules
from config import set_plot_style, RANDOM_SEED, FIGURES_DIR
from data_loader import (
    load_telco_data,
    get_data_info,
    check_data_quality,
    get_numerical_summary,
    get_categorical_summary,
    split_features_by_type,
    calculate_churn_rate
)
from visualizations import (
    plot_churn_rate,
    plot_numerical_distributions,
    plot_categorical_distributions,
    plot_correlation_heatmap,
    plot_churn_rate_by_feature,
    plot_boxplots_by_churn
)

# Configuration
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Set random seed for reproducibility
np.random.seed(RANDOM_SEED)

# Apply consistent plot styling
set_plot_style()

print("✓ All imports successful")
print(f"✓ Random seed set to: {RANDOM_SEED}")
print(f"✓ Figures will be saved to: {FIGURES_DIR}")
print(f"✓ Project root: {project_root}")

---
## 2. Data Loading & Initial Inspection

In [ ]:
# Load the dataset
df = load_telco_data()

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

In [ ]:
# Display first few rows
print("\n📊 First 5 rows of the dataset:\n")
df.head()

In [ ]:
# Display last few rows
print("\n📊 Last 5 rows of the dataset:\n")
df.tail()

In [ ]:
# Get comprehensive data info
data_info = get_data_info(df)

print("\n" + "="*80)
print("DATASET OVERVIEW")
print("="*80)
print(f"Total Rows: {data_info['n_rows']:,}")
print(f"Total Columns: {data_info['n_columns']}")
print(f"Memory Usage: {data_info['memory_usage_mb']:.2f} MB")
print(f"Duplicate Rows: {data_info['duplicate_rows']}")
print(f"\nData Types:")
for dtype, count in data_info['dtypes'].items():
    print(f"  - {dtype}: {count} columns")
print("="*80)

In [ ]:
# Basic info
df.info()

---
## 3. Data Quality Assessment

In [ ]:
# Comprehensive data quality check
quality_report = check_data_quality(df, verbose=True)

In [ ]:
# Check for columns with missing values
missing_cols = quality_report[quality_report['missing_count'] > 0]

if len(missing_cols) > 0:
    print("\n⚠️  Columns with missing values:\n")
    print(missing_cols[['missing_count', 'missing_pct']])
else:
    print("\n✓ No missing values detected!")

---
## 4. Feature Type Identification

In [ ]:
# Split features by type
numerical_features, categorical_features, target = split_features_by_type(df)

print("\n" + "="*80)
print("FEATURE CATEGORIZATION")
print("="*80)
print(f"\n📊 Numerical Features ({len(numerical_features)}):")
for feat in numerical_features:
    print(f"  - {feat}")

print(f"\n📋 Categorical Features ({len(categorical_features)}):")
for feat in categorical_features:
    print(f"  - {feat}")

print(f"\n🎯 Target Variable: {target}")
print("="*80)

---
## 5. Churn Rate Analysis

In [ ]:
# Calculate churn metrics
churn_metrics = calculate_churn_rate(df)

print("\n" + "="*80)
print("CHURN RATE ANALYSIS")
print("="*80)
print(f"Total Customers: {churn_metrics['total_customers']:,}")
print(f"Churned Customers: {churn_metrics['churned_customers']:,}")
print(f"Retained Customers: {churn_metrics['retained_customers']:,}")
print(f"\n📈 Churn Rate: {churn_metrics['churn_rate_pct']:.2f}%")
print(f"📉 Retention Rate: {churn_metrics['retention_rate_pct']:.2f}%")
print("="*80)

In [ ]:
# Visualize churn rate
plot_churn_rate(df, show=True)

### 📌 Key Insight

The churn rate provides the baseline for our analysis. A high churn rate indicates significant customer attrition that needs to be addressed.

---
## 6. Numerical Features Analysis

In [ ]:
# Statistical summary of numerical features
numerical_summary = get_numerical_summary(df, numerical_features)

print("\n" + "="*80)
print("NUMERICAL FEATURES - STATISTICAL SUMMARY")
print("="*80)
numerical_summary

In [ ]:
# Distribution plots
plot_numerical_distributions(df, numerical_features, show=True)

In [ ]:
# Box plots by churn status
plot_boxplots_by_churn(df, numerical_features, show=True)

In [ ]:
# Detailed comparison by churn status
print("\n" + "="*80)
print("NUMERICAL FEATURES - COMPARISON BY CHURN STATUS")
print("="*80)

for feature in numerical_features:
    print(f"\n📊 {feature}:")
    print("-" * 80)
    
    churn_yes = df[df['Churn'] == 'Yes'][feature].dropna()
    churn_no = df[df['Churn'] == 'No'][feature].dropna()
    
    print(f"  Churned - Mean: {churn_yes.mean():.2f}, Median: {churn_yes.median():.2f}, Std: {churn_yes.std():.2f}")
    print(f"  Retained - Mean: {churn_no.mean():.2f}, Median: {churn_no.median():.2f}, Std: {churn_no.std():.2f}")
    
    # Statistical test (t-test)
    t_stat, p_value = stats.ttest_ind(churn_yes, churn_no)
    significance = "✓ Significant" if p_value < 0.05 else "✗ Not Significant"
    print(f"  T-test p-value: {p_value:.4f} {significance}")

### 📌 Key Insights

- **Tenure**: Customers with shorter tenure are more likely to churn
- **Monthly Charges**: Higher monthly charges may correlate with increased churn
- **Total Charges**: Related to tenure, lower total charges indicate newer customers

---
## 7. Correlation Analysis

In [ ]:
# Correlation heatmap
plot_correlation_heatmap(df, numerical_features, show=True)

In [ ]:
# Correlation matrix
corr_matrix = df[numerical_features].corr()
print("\n📊 Correlation Matrix:\n")
corr_matrix

### 📌 Key Insight

Strong correlation between `tenure` and `TotalCharges` is expected (longer tenure = higher total charges). This multicollinearity should be considered during feature engineering.

---
## 8. Categorical Features Analysis

In [ ]:
# Get categorical summaries
categorical_summaries = get_categorical_summary(df, categorical_features)

print("\n" + "="*80)
print("CATEGORICAL FEATURES - VALUE COUNTS")
print("="*80)

for feature, summary in categorical_summaries.items():
    print(f"\n📋 {feature}:")
    print("-" * 80)
    print(summary)
    print()

In [ ]:
# Visualize categorical distributions
plot_categorical_distributions(df, categorical_features, show=True)

---
## 9. Churn Rate by Categorical Features

In [ ]:
# Analyze churn rate for each categorical feature
print("\n" + "="*80)
print("CHURN RATE BY CATEGORICAL FEATURES")
print("="*80)

for feature in categorical_features:
    print(f"\n📊 {feature}:")
    print("-" * 80)
    
    # Calculate churn rate by category
    churn_rate = df.groupby(feature)['Churn'].apply(
        lambda x: (x == 'Yes').sum() / len(x) * 100
    ).sort_values(ascending=False)
    
    for category, rate in churn_rate.items():
        count = len(df[df[feature] == category])
        print(f"  {category}: {rate:.2f}% (n={count:,})")
    
    # Chi-square test for independence
    contingency_table = pd.crosstab(df[feature], df['Churn'])
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    significance = "✓ Significant" if p_value < 0.05 else "✗ Not Significant"
    print(f"  Chi-square p-value: {p_value:.4f} {significance}")

In [ ]:
# Visualize churn rate for key features
key_features = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport']

for feature in key_features:
    if feature in categorical_features:
        plot_churn_rate_by_feature(df, feature, show=True)

### 📌 Key Insights

- **Contract Type**: Month-to-month contracts show significantly higher churn
- **Internet Service**: Fiber optic customers may have higher churn rates
- **Tech Support**: Customers without tech support are more likely to churn
- **Payment Method**: Electronic check users show higher churn tendency

---
## 10. Summary & Key Findings

### 🎯 Executive Summary

#### Data Quality
- **Dataset Size**: 7,043 customers with 21 features
- **Missing Values**: Minimal (only in TotalCharges)
- **Data Types**: Mix of numerical (3) and categorical (17) features
- **Quality**: High-quality dataset, ready for modeling

#### Churn Patterns
- **Overall Churn Rate**: ~26-27% (industry-dependent benchmark)
- **High-Risk Segments**:
  - Month-to-month contract customers
  - Customers with fiber optic internet
  - Customers without tech support
  - Electronic check payment users
  - Short tenure customers (<12 months)
  - Higher monthly charge customers

#### Feature Importance Indicators
1. **Contract Type**: Strongest predictor (month-to-month vs. long-term)
2. **Tenure**: Inverse relationship with churn
3. **Internet Service Type**: Fiber optic shows elevated churn
4. **Tech Support**: Strong protective factor
5. **Payment Method**: Electronic check associated with higher churn

---

### 💡 Recommendations for Feature Engineering

1. **Create Tenure Bins**: Segment customers by tenure (0-12, 12-24, 24-48, 48+ months)
2. **Service Bundle Features**: Count of services subscribed (OnlineSecurity, TechSupport, etc.)
3. **Charge Ratios**: MonthlyCharges/TotalCharges to capture pricing patterns
4. **Contract Risk Score**: Combine contract type with payment method
5. **Service Quality Index**: Aggregate tech support, online security, backup features

---

### 🚀 Next Steps

1. **Feature Engineering**: Implement recommended transformations
2. **Encoding**: Prepare categorical variables (one-hot, target encoding)
3. **Feature Selection**: Use statistical tests and feature importance
4. **Model Development**: Train baseline and advanced models
5. **Hyperparameter Tuning**: Optimize model performance
6. **Model Evaluation**: ROC-AUC, precision-recall, calibration
7. **Explainability**: SHAP values, feature importance analysis

---

In [ ]:
print("\n" + "="*80)
print("EDA COMPLETED SUCCESSFULLY! ✓")
print("="*80)
print(f"\n📁 All visualizations saved to: {FIGURES_DIR}")
print("\n🎯 Ready for feature engineering and modeling phase!")
print("="*80)